# End-to-End Intelligent DQ Validation
This is the canonical runner. It rebuilds the local hybrid context index, verifies the configured LLM before warehouse access, executes validation and agentic RCA, and writes one report workbook.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'config').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from dq_agent.config import load_app_config
from dq_agent.context_store import make_context_retriever
from dq_agent.llm import make_llm_adapter
from dq_agent.workflow import DQWorkflow

In [ ]:
RUN_ID = 'validation_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_VALIDATION = False  # Set True only when LLM and warehouse credentials are ready
STOP_AFTER = 'reports'  # metadata, inference, rules, execution, rca, or reports
config = load_app_config(ROOT)
output_dir = config.path(config.project.outputs_dir) / RUN_ID
log_dir = config.path(config.project.logs_dir) / RUN_ID

In [ ]:
retriever = make_context_retriever(config)
sync = retriever.sync()
print('Context sync:', sync)
display(pd.DataFrame(retriever.search('orders customers primary key relationships measures', limit=8)))

In [ ]:
if RUN_VALIDATION:
    health = make_llm_adapter(config).health_check()
    print('LLM preflight:', health.model_dump())
else:
    print('Preflight deferred. Enabling RUN_VALIDATION makes it mandatory and it runs before warehouse access.')

In [ ]:
manifest = None
if RUN_VALIDATION:
    manifest = DQWorkflow(ROOT).run(run_id=RUN_ID, stop_after=STOP_AFTER)
    display(pd.DataFrame(manifest.get('tables', [])))
    print('Run status:', manifest['status'])
    print('Report:', output_dir / 'dq_validation_report.xlsx')
    print('Logs:', log_dir)
    print('Approval workbook:', manifest.get('approval_workbook') or 'none')
else:
    print('Validation skipped. Inspect retrieved context, then set RUN_VALIDATION=True.')

In [ ]:
if manifest:
    checkpoint = log_dir / 'checkpoint.json'
    events = log_dir / 'events.jsonl'
    if checkpoint.exists(): display(json.loads(checkpoint.read_text(encoding='utf-8')))
    if events.exists():
        display(pd.DataFrame([json.loads(line) for line in events.read_text(encoding='utf-8').splitlines()[-20:]]))

## Artifacts
The run produces exactly `outputs/<run_id>/dq_validation_report.xlsx`. SQL, manifests, checkpoints, diagnostic evidence, and logs stay under `logs/<run_id>/`. If human decisions are needed, the run creates at most one workbook under `approvals/pending/`; move it to `reviewed/` before running notebook `02`.